# Projeto 01 — Agente de Rotas para Comida Local (IA 2026-27)

**Inteligência Artificial — Licenciatura em Engenharia de Sistemas Informáticos (IPCA / EST)**

### Contexto
Uma plataforma digital liga **produtores** locais, **clientes** e **centros de recolha (armazéns)**.
Cada cliente faz uma encomenda e escolhe um **local de entrega**. Os produtos são recolhidos nos produtores,
passam obrigatoriamente pelo armazém (`producer > warehouse > delivery_location`) e são depois entregues,
respeitando:
* a **janela de disponibilidade** do produto no produtor (`availability_start / availability_end`);
* a **janela de entrega** no local de destino (`delivery_start / delivery_end`);
* a **capacidade** dos veículos.

O objetivo do projeto é desenvolver um agente que escolha as melhores rotas de recolha e entrega usando
**pesquisa informada** (Greedy, A*), **pesquisa local** (Hill-Climbing) e **CSP** (biblioteca `python-constraint`).

### Equipa (Grupo G##)
| Nome | Número |
|------|--------|
| ... | ... |
| ... | ... |
| ... | ... |
| ... | ... |

### Este notebook — Entrega 1: Parser (`01_eda_tiny.ipynb`)
O parser lê os ficheiros CSV do dataset e carrega-os para variáveis Python, seguindo a mesma lógica do
notebook `Notebook_91_CSP_CourseTT_PC_w_data.ipynb`: uma função `parse_...` devolve um dicionário
`parsed_data` com uma secção por tipo de dados; depois constroem-se as estruturas auxiliares que os
algoritmos vão usar.

O **único resultado mostrado** é o número de linhas lidas de cada ficheiro e carregadas para a memória
(secção 3). A análise completa, com gráficos, está no notebook `01_eda_tiny_comp.ipynb`.

In [1]:
%%capture
# Instalar as bibliotecas usadas no projeto (%%capture esconde o texto da instalação)
# (python-constraint2 é a versão mantida da biblioteca python-constraint: importa-se com "from constraint import *")
%pip install pandas python-constraint2

In [2]:
# Importar bibliotecas
import glob
import os
import zipfile
from datetime import datetime

import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

---
## 1. Escolha do dataset

Existem dois datasets (fornecidos no Moodle em ZIP):
* `tiny`  — pequeno e simples;
* `small` — maior e mais complexo.

Basta alterar a variável `DATASET` para correr todo o notebook sobre o outro dataset.
Se a pasta ainda não existir, o ZIP correspondente é extraído automaticamente para a pasta `data/`.

In [3]:
DATASET = 'tiny'      # 'tiny' ou 'small'
DATA_DIR = 'data'

def get_dataset_folder(dataset, data_dir=DATA_DIR):
    # Return the folder of the dataset, extracting the ZIP file if needed
    folders = glob.glob(os.path.join(data_dir, f'data_{dataset}_*'))
    if not folders:
        zips = glob.glob(f'data_{dataset}_*.zip')
        if not zips:
            raise FileNotFoundError(f'ZIP do dataset "{dataset}" não encontrado')
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(data_dir)
        folders = glob.glob(os.path.join(data_dir, f'data_{dataset}_*'))
    return sorted(folders)[0]

folder = get_dataset_folder(DATASET)

---
## 2. Parser

Tal como no notebook de referência (`parse_file` → `parsed_data`), a função `parse_dataset`:
1. lê cada ficheiro CSV com `pandas.read_csv` para uma **tabela** (`DataFrame`), já com as datas convertidas;
2. guarda as tabelas no dicionário `tables` (útil para os resumos);
3. converte cada tabela num dicionário indexado pela **chave primária (PK)** e guarda-o em `parsed_data`
   (útil para os algoritmos: `parsed_data['products'][3]` devolve o produto com `product_id = 3`).

| Secção | Ficheiro | Chave | Conteúdo |
|---|---|---|---|
| `rules` | rules.csv | `rule` | parâmetros do problema (horizonte, tempos de serviço, ...) |
| `localities` | localities.csv | `locality_id` | freguesias geocodificadas |
| `producers` | producers.csv | `producer_id` | produtores |
| `products` | products.csv | `product_id` | produtos de cada produtor |
| `centers` | collection_centers.csv | `center_id` | armazéns (base dos veículos) |
| `vehicles` | vehicles.csv | `vehicle_id` | frota |
| `customers` | customers.csv | `customer_id` | clientes |
| `delivery_locations` | delivery_locations.csv | `delivery_location_id` | locais de entrega |
| `orders` | orders.csv | `order_id` | encomendas |
| `order_items` | order_items.csv | `order_item_id` | linhas de encomenda com as janelas temporais |
| `locations` | locations.csv | `location_id` | paragens de routing (produtores, armazém, entregas) |
| `distance_matrix` | distance_matrix.csv | `(origem, destino)` | distância (km) e duração (min) |

In [4]:
# Information about each CSV file: section -> (file name, primary key, columns with dates)
FILES = {
    'localities':         ('localities.csv',         'locality_id',          []),
    'producers':          ('producers.csv',          'producer_id',          []),
    'products':           ('products.csv',           'product_id',           []),
    'centers':            ('collection_centers.csv', 'center_id',            []),
    'vehicles':           ('vehicles.csv',           'vehicle_id',           []),
    'customers':          ('customers.csv',          'customer_id',          []),
    'delivery_locations': ('delivery_locations.csv', 'delivery_location_id', []),
    'orders':             ('orders.csv',             'order_id',             ['order_datetime']),
    'order_items':        ('order_items.csv',        'order_item_id',
                           ['availability_start', 'availability_end', 'delivery_start', 'delivery_end']),
    'locations':          ('locations.csv',          'location_id',          []),
    'distance_matrix':    ('distance_matrix.csv',    None,                   []),
}


def parse_rule_value(value):
    # Convert a rule value to int or float when possible, otherwise keep the text
    try:
        return int(value)
    except ValueError:
        pass
    try:
        return float(value)
    except ValueError:
        return value


def parse_dataset(folder):
    # Function to read and parse all the CSV files of a dataset
    tables = {}        # section -> pandas DataFrame
    data = {}          # section -> {primary key: record}

    # Rules: rule -> value
    rules_df = pd.read_csv(os.path.join(folder, 'rules.csv'), dtype=str)
    data['rules'] = {}
    for rule, value in zip(rules_df['rule'], rules_df['value']):
        data['rules'][rule] = parse_rule_value(value)
    tables['rules'] = rules_df

    # Entity tables
    for section, (file_name, pk, date_columns) in FILES.items():
        df = pd.read_csv(os.path.join(folder, file_name), parse_dates=date_columns)
        if section == 'vehicles':
            # The column "refrigerated" is 1/0 in the file: convert it to True/False
            df['refrigerated'] = df['refrigerated'].astype(bool)
        tables[section] = df
        if pk is None:
            continue
        if df[pk].duplicated().any():
            raise ValueError(f'{file_name}: chave primária repetida')
        # DataFrame -> dictionary {primary key: {column: value}}
        data[section] = df.set_index(pk, drop=False).to_dict('index')

    # Distance matrix: (origin, destination) -> {'distance_km': ..., 'duration_min': ...}
    data['distance_matrix'] = {}
    for row in tables['distance_matrix'].itertuples():
        key = (row.origin_location_id, row.destination_location_id)
        data['distance_matrix'][key] = {'distance_km': row.distance_km, 'duration_min': row.duration_min}

    return data, tables


parsed_data, tables = parse_dataset(folder)

### 2.1 Estruturas auxiliares para os algoritmos

Tal como no notebook de referência (onde se constroem `weekcls`, `course_list`, `lesson_list`, `study_plan`),
transformamos os dados carregados nas estruturas que os algoritmos de pesquisa e o CSP vão usar:

* **Tempo em minutos**: todas as datas são convertidas em minutos desde `start_date 00:00`
  (dia 1 = 0..1439, dia 2 = 1440..2879). Assim, as janelas temporais passam a ser intervalos de inteiros.
* **Paragens (`location_id`)**: cada produtor, armazém e local de entrega corresponde a uma paragem de
  `locations.csv`; as tabelas `loc_of_producer`, `loc_of_center` e `loc_of_delivery` fazem esta tradução.
* **`dist` / `dur`**: dicionários `(origem, destino) -> km / min` para consulta rápida.
* **`requests`**: um pedido de transporte por linha de encomenda (recolha no produtor → armazém → entrega).
* **`vehicles`**: capacidade e paragem de partida/chegada (armazém) de cada veículo.

In [5]:
# Planning horizon and rules
rules = parsed_data['rules']
T0 = datetime.strptime(str(rules['start_date']), '%Y-%m-%d')   # time origin (minute 0)
HORIZON_MIN = rules['horizon_days'] * 24 * 60
SERVICE_MIN = rules['stop_service_min']            # service time at each pickup / delivery stop
HANDLING_MIN = rules['warehouse_handling_min']     # unload / load time at the warehouse
MAX_DWELL_MIN = rules['max_warehouse_dwell_h'] * 60

def to_min(dt):
    # Minutes since the beginning of the planning horizon
    return int((dt - T0).total_seconds() // 60)

def fmt_min(m):
    # Minutes since T0 -> 'D1 08:30'
    d, r = divmod(int(m), 1440)
    return f'D{d + 1} {r // 60:02d}:{r % 60:02d}'

# Routing stops: entity id -> location_id
loc_of_producer, loc_of_center, loc_of_delivery = {}, {}, {}
for lid, loc in parsed_data['locations'].items():
    {'producer': loc_of_producer,
     'collection_center': loc_of_center,
     'delivery_location': loc_of_delivery}[loc['location_type']][loc['ref_id']] = lid

stops = sorted(parsed_data['locations'])
dist = {k: v['distance_km'] for k, v in parsed_data['distance_matrix'].items()}
dur = {k: v['duration_min'] for k, v in parsed_data['distance_matrix'].items()}
for s in stops:                         # travelling to the same stop costs nothing
    dist[(s, s)] = dur[(s, s)] = 0.0

# Vehicles: capacity and depot (warehouse stop)
vehicles = {vid: {'capacity': v['capacity_kg'],
                  'depot': loc_of_center[v['center_id']],
                  'refrigerated': v['refrigerated']}
            for vid, v in parsed_data['vehicles'].items()}

# Requests: one per order item (pickup at producer -> warehouse -> delivery location)
requests = {}
for iid, it in parsed_data['order_items'].items():
    order = parsed_data['orders'][it['order_id']]
    product = parsed_data['products'][it['product_id']]
    requests[iid] = {
        'order': it['order_id'],
        'product': product['name'],
        'producer': product['producer_id'],
        'pickup': loc_of_producer[product['producer_id']],
        'delivery': loc_of_delivery[order['delivery_location_id']],
        'weight': it['weight_kg'],
        'av_start': to_min(it['availability_start']),
        'av_end': to_min(it['availability_end']),
        'del_start': to_min(it['delivery_start']),
        'del_end': to_min(it['delivery_end']),
    }

# Requests grouped by order and by producer
order_requests = {}
producer_requests = {}
for iid, r in requests.items():
    order_requests.setdefault(r['order'], []).append(iid)
    producer_requests.setdefault(r['producer'], []).append(iid)

WAREHOUSE = loc_of_center[1]

---
## 3. Linhas lidas e carregadas para a memória

Para cada ficheiro CSV mostra-se o número de linhas lidas e carregadas em `parsed_data`.
Antes de imprimir, confirma-se que cada tabela lida (`tables`) tem o mesmo número de linhas que os
registos carregados (`parsed_data`); se não tiver, o `assert` pára o notebook com um erro.

In [6]:
# Name of the CSV file of each section
file_names = {'rules': 'rules.csv'}
for section, (file_name, pk, date_columns) in FILES.items():
    file_names[section] = file_name

print(f'Dataset: {DATASET}\n')
total = 0
for section, records in parsed_data.items():
    # Check that every row read from the file was loaded into memory
    assert len(records) == len(tables[section]), f'{section}: nem todas as linhas foram carregadas'
    print(f'{file_names[section]:<26} {len(records):>4} linhas')
    total += len(records)
print(f"{'Total':<26} {total:>4} linhas")

Dataset: tiny

rules.csv                    10 linhas
localities.csv               49 linhas
producers.csv                 3 linhas
products.csv                 15 linhas
collection_centers.csv        1 linhas
vehicles.csv                  2 linhas
customers.csv                12 linhas
delivery_locations.csv        5 linhas
orders.csv                   12 linhas
order_items.csv              40 linhas
locations.csv                 9 linhas
distance_matrix.csv          72 linhas
Total                       230 linhas
